# Import

In [186]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline 
# pour afficher facilement les graphiques 
from urllib.request import urlopen
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import os
import re
from pathlib import Path
from nltk.stem import SnowballStemmer
import spacy


# Variables 

In [187]:
# configuration des chemins 
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().parent

BULLETINS = BASE_DIR / "BULLETINS"
DATA = BASE_DIR / "data"
OUTPUT = BASE_DIR / "output"

# Fonction d'extraction 

In [188]:
def extraction_metadonnees(requete):
    metadonnees = {}
    # contriantes temporelle
    pattern_date =  r"entre le \d{1,2} \w+ \d{4} et \d{1,2} \w+ \d{4}|" \
                    r"entre \d{1,2}/\s*\d{1,2}/\s*\d{4} et \d{1,2}/\s*\d{1,2}/\s*\d{4}|"\
                    r"entre \d{4} et \d{4}|" \
                    r"après \d{1,2}/\s*\d{1,2}/\s*\d{4} |"\
                    r"de \d{4} |" \
                    r"au mois de \w+ \d{4}|"\
                    r"du mois de \w+ \d{4}|"\
                    r"au mois de \w+ |"\
                    r"en \w+ \d{4}|"\
                    r"en \d{4}|"\
                    r"à partir de \d{4}|"\
                    r"à partir de \w+ \d{4}|"\
                    r"de l'année \d{4}|"\
                    r"après janvier \d{4}|"\
                    r"du \d{1,2} \w+ \d{4}"\

    resultat_date = re.findall(pattern_date,  requete, re.IGNORECASE)
    metadonnees['dates'] = resultat_date[0] if len(resultat_date) > 0 else ""

    # rubrique 
    pattern_rubrique = r"(focus|horizons enseignement|en direct des laboratoires|a lire|actualités innovations|événement)"
    resultat_rubrique = re.findall(pattern_rubrique, requete, re.IGNORECASE)
    metadonnees['rubrique'] = resultat_rubrique[0].split() if len(resultat_rubrique) > 0 else ""

    # filtre structurel
    pattern_structurel = r"avec des images|contenant une imagesans|qui \w+ des images|sans image|contenant le mot \w+"
    resultat_structurel = re.findall(pattern_structurel,  requete, re.IGNORECASE)
    metadonnees['structurel'] = resultat_structurel[0].split() if len(resultat_structurel) > 0 else ""

    #operateur
    pattern_operateur = r"et|ou"
    resultat_operateur = re.findall(pattern_operateur,  requete, re.IGNORECASE)
    metadonnees['operateur'] = list(set(resultat_operateur))

    #operateur exclusion 
    pattern_operateur_not = r"(?:mais pas|sans) \w+(?:\s\w+)*"
    resultat_operateur_not = re.findall(pattern_operateur_not,  requete, re.IGNORECASE)
    metadonnees['exclusion'] = resultat_operateur_not[0].split() if len(resultat_operateur_not) > 0 else ""

    print("metadonnees ---> ✅ ")
    print(
    "date :", metadonnees['dates'],
    "\n\trubrique :", metadonnees['rubrique'],
    "\n\toperateur :", metadonnees['operateur'],
    "\n\texclusion :", metadonnees['exclusion'],
    "\n\tdate_min :", metadonnees['structurel'],
    )

    return metadonnees  


In [189]:
def reste(requete, metadonnees):
    mots_restant = requete
    for data in metadonnees :
        data = metadonnees[data]
        if isinstance(data,list):
            for element in data :
                mots_restant = mots_restant.replace(element,"")
        else :
            mots_restant = mots_restant.replace(data,"")

    return mots_restant 

# traitement des dates

In [190]:
mois_dict = {
    "janvier": "01", "février": "02", "mars": "03",
    "avril": "04", "mai": "05", "juin": "06",
    "juillet": "07", "août": "08", "septembre": "09",
    "octobre": "10", "novembre": "11", "décembre": "12"
}

def convertir_date(date_str):
    """
    transforme 10 avril 2012 en 10/04/2012
    
    """
    parts = date_str.split()

    jour = parts[0].zfill(2)
    mois_str = mois_dict[parts[1]]
    annee = parts[2]

    return f"{jour}/{mois_str}/{annee}"

def traiter_expression(expr):
    """
    Le but de la fonction c'est d'avoir des dates in et out pour l'obtention de l'intervale sous un format datetime comprehensible par python 
    """
    date_min = "01/01/2011"
    date_max = "01/01/2015"

    # initialisation 
    dates = {}
    dates['in'] = date_min
    dates['out'] = date_max

    # mise à jour
    pattern = r"entre le (\d{1,2} \w+ \d{4}) et (\d{1,2} \w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = convertir_date(match.group(1))
        date2 = convertir_date(match.group(2))
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"entre( \d{1,2}/\s*\d{1,2}/\s*\d{4}) et (\d{1,2}/\s*\d{1,2}/\s*\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = match.group(2)
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"entre (\d{4}) et (\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = match.group(2)
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"après (\d{1,2}/\s*\d{1,2}/\s*\d{4}) "
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = date_max
        dates['in'] = date1
        dates['out'] = date2
    
    pattern = r"à partir de (\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = date_max
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"au mois de (\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = f"01/{int(mois_min)+1}/{match.group(1).split()[1]}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"du mois de (\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = f"01/{int(mois_min)+1}/{match.group(1).split()[1]}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"en (\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = f"01/{int(mois_min)+1}/{match.group(1).split()[1]}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"(?:en|de|de l'année)\s+(\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = f"01/01/{match.group(1)}"
        date2 = f"31/12/{match.group(1)}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"(?:à partir de|après)\s+(\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = date_max
        dates['in'] = date1
        dates['out'] = date2
    
    pattern = r"du (\d{1,2} \w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = convertir_date(match.group(1))
        date2 = convertir_date(match.group(1))
        dates['in'] = date1
        dates['out'] = date2
    
    
    return dates



In [191]:
expr = "au mois de avril 2012"
output = traiter_expression(expr)
print(output)

{'in': '01/4/2012', 'out': '01/5/2012'}


# obtention de la structure 

In [192]:
def traitement_requete(requete):
    metadonnees = extraction_metadonnees(requete)
    mots_restant = reste(requete, metadonnees)
    mots_cles = mots_restant.split()
    dates = traiter_expression(metadonnees['dates'])
    mots = ["mais","pas","sans"]
    for mot in mots : 
        mots_exclus = metadonnees['exclusion'].replace(mot,"") if len(metadonnees['exclusion']) > 0 else ""

    structuration = {
        'mots_cles': mots_cles,
        'rubrique' : metadonnees['rubrique'], 
        'operateur' : [op.upper() for op in metadonnees['operateur']],
        'exclusion' : mots_exclus.split(),
        'date_min' : dates['in'], 
        'date_max' : dates['out']
    }
    print("structure --- ✅ ")
    print(
    "\tmots_cles :", structuration['mots_cles'],
    "\n\trubrique :", structuration['rubrique'],
    "\n\toperateur :", structuration['operateur'],
    "\n\texclusion :", structuration['exclusion'],
    "\n\tdate_min :", structuration['date_min'],
    "\n\tdate_max :", structuration['date_max']
    )
    return structuration

In [193]:
requete = "Je voudraisles articles qui datent du 1 décembre 2012 et dont la rubrique est Actualités Innovations"
traitement_requete(requete)

metadonnees ---> ✅ 
date : du 1 décembre 2012 
	rubrique : ['Actualités', 'Innovations'] 
	operateur : ['ou', 'et'] 
	exclusion :  
	date_min : 
structure --- ✅ 
	mots_cles : ['Je', 'vdraisles', 'articles', 'qui', 'datent', 'dont', 'la', 'rubrique', 'est'] 
	rubrique : ['Actualités', 'Innovations'] 
	operateur : ['OU', 'ET'] 
	exclusion : [] 
	date_min : 01/12/2012 
	date_max : 01/12/2012


{'mots_cles': ['Je',
  'vdraisles',
  'articles',
  'qui',
  'datent',
  'dont',
  'la',
  'rubrique',
  'est'],
 'rubrique': ['Actualités', 'Innovations'],
 'operateur': ['OU', 'ET'],
 'exclusion': [],
 'date_min': '01/12/2012',
 'date_max': '01/12/2012'}

# td06

## fonction universelles 

In [ ]:
def charger_index(fichier):
    """
    transforme les fichiers inverses en dictionnaire 

    """

    index = {}

    with open(fichier, "r", encoding="utf-8") as f:

        for ligne in f:

            ligne = ligne.strip()

            if not ligne:
                continue

            # séparer mot et reste
            mot, docs_str = ligne.split("\t")

            docs = {}

            # séparer les doc:freq
            for element in docs_str.split():

                doc, freq = element.split(":")

                docs[doc] = int(freq)

            index[mot] = docs

    return index


def construire_requete(structure):
    #initialisation de l'operateur 

    mots = structure['mots_cles']
    operateur = structure['operateur'][0].upper() if len(structure['operateur']) > 0 else ['ET']
    exclus = structure.get('exclus', [])
    sep = " OR " if operateur == "OU" else " AND "
    base = sep.join(mots)
    requete = base
    if exclus:
        excl = " AND ".join(f"NOT {m}" for m in exclus)
        requete = f"{base} AND {excl}"
    
    print("construction de la requete --> ✅  ",requete)
    return requete

def executer_requete(requete, index):

    # détecter opérateur principal
    if " AND " in requete:
        morceaux = requete.split(" AND ")
        operateur = "AND"
    elif " OR " in requete:
        morceaux = requete.split(" OR ")
        operateur = "OR"
    else:
        morceaux = [requete]
        operateur = None

    inclus = []
    exclus = []

    # séparer inclus / exclus
    for m in morceaux:
        m = m.strip()
        if m.startswith("NOT "):
            exclus.append(m.replace("NOT ", ""))
        else:
            inclus.append(m)

    # sets de documents
    docs_inclus = [set(index[m].keys()) for m in inclus if m in index]
    docs_exclus = set()
    for m in exclus:
        if m in index:
            docs_exclus |= set(index[m].keys())

    if not docs_inclus:
        return {}
    print ("\tdocs_inclus -->", docs_inclus )
    # AND / OR
    if operateur == "AND":
        docs_final = set.intersection(*docs_inclus)
    elif operateur == "OR":
        docs_final = set.union(*docs_inclus)
    else:
        docs_final = docs_inclus[0]

    # appliquer exclusion
    docs_final = docs_final - docs_exclus

    # score (somme des fréquences)
    resultat = {}
    for doc in docs_final:
        score = 0
        for mot in inclus:
            if mot in index and doc in index[mot]:
                score += index[mot][doc]
        resultat[doc] = score

    return resultat

def intersection_multiple(dicts):
    if not dicts:
        return {}

    # intersection des clés
    docs_communs = set(dicts[0].keys())

    for d in dicts[1:]:
        docs_communs &= set(d.keys())

    # somme des fréquences
    resultat = {}

    for doc in docs_communs:
        resultat[doc] = sum(d[doc] for d in dicts)

    return resultat

In [195]:
# requete = " entre le 4 avril 2011 et 12 décembre 2011 écoles entreprise  Actualités Innovations"
# inverse_texte = charger_index(DATA/"inverse_texte.txt")
# strcture = traitement_requete(requete)
# req = construire_requete(strcture)
# resultat = executer_requete(req,inverse_texte)
# print(resultat)

In [196]:
dictionnaire = charger_index(DATA/"inverse_date.txt")
for cle, valeur in list(dictionnaire.items())[:1]:
    for x,y in list(valeur.items())[:2]:
        print(cle, x, y)

2011/06/21 67068 1
2011/06/21 67071 1


{'mots_cles': ['Je',
  'voudraisles',
  'articles',
  'qui',
  'datent',
  'dont',
  'la',
  'rubrique',
  'est'],
 'rubrique': ['Actualités Innovations'],
 'operateur': ['ET '],
 'date_min': '01/12/2012',
 'date_max': '01/12/2012'}

## moteur.py 

In [197]:

def moteur(requete):

    # traitement de la requete 
    structure = traitement_requete(requete)

    # recupération des fichier inverse 
    inverse_date = charger_index(DATA/"inverse_date.txt")
    inverse_rubrique = charger_index(DATA/"inverse_rubrique.txt")
    inverse_texte = charger_index(DATA/"inverse_texte.txt")
    inverse_titre = charger_index(DATA/"inverse_titre.txt")

    # recuperation des bons document
    # initialisation 
    docs_date = {}
    docs_rubrique = {}
    docs_texte = {}
    docs_titre = {}
    # 1. filtre sur les dates
    if inverse_date :
        for cle in inverse_date :
            date_doc = pd.to_datetime(cle, format="mixed", dayfirst=True, errors="coerce")
            date_min = pd.to_datetime(structure['date_min'], format="mixed", dayfirst=True, errors="coerce")
            date_max = pd.to_datetime(structure['date_max'], format="mixed", dayfirst=True, errors="coerce")
            if date_doc is not pd.NaT and date_min <= date_doc <= date_max:

                for doc, freq in list(inverse_date[cle].items()) : 
                    docs_date[doc] = freq
        print("filtre date ok ✅ ")
    # 2. filtre rubrique (hyphothèse il y a une seul rubrique dans la requette )
    if inverse_rubrique :
        for cle in inverse_rubrique :
            for mot in structure['rubrique'] :
                if mot == cle :
                    for doc, freq in list(inverse_rubrique[cle].items()) : 
                        docs_rubrique[doc] = freq
        print("filtre rubrique ok ✅ ")
    # 3. filtre texte
    if inverse_texte :
        req = construire_requete(structure)
        docs_texte = executer_requete(req,inverse_texte)
        print("filtre texte ok ✅ ")

    if inverse_titre :
        req = construire_requete(structure)
        docs_titre = executer_requete(req,inverse_titre)
        print("filtre titre ok ✅ ")
        
    # on recupère les dictionnaires non vide 
    liste_docs = [d for d in [docs_date, docs_rubrique, docs_texte, docs_titre] if d]
    resultat_final = intersection_multiple(liste_docs)

    # classe par ordre décroissant 
    resultat_final = dict(sorted(resultat_final.items(), key=lambda x: x[1], reverse=True))
    
    # affichage des resultats 
    print("documents chercher ---> ✅  ")
    print("\t",resultat_final)
    return resultat_final

                
requete = "entreprise de l'année 2013"
resultat = moteur(requete)


metadonnees ---> ✅ 
date : de l'année 2013 
	rubrique :  
	operateur : [] 
	exclusion :  
	date_min : 
structure --- ✅ 
	mots_cles : ['entreprise'] 
	rubrique :  
	operateur : [] 
	exclusion : [] 
	date_min : 01/01/2013 
	date_max : 31/12/2013
filtre date ok ✅ 
filtre rubrique ok ✅ 
construction de la requete --> ✅   entreprise
docs_inclus --> [{'70165', '74453', '71840', '74451', '74747', '74450', '73435', '76511', '68274', '72630', '74748', '76207', '73878', '75063', '76212', '69815', '73187', '73875', '73184', '70422', '68384', '68273', '71614', '74167', '76507', '70166', '72634', '75065', '73686', '70421', '76509', '68643', '73879', '71841', '72933', '71360', '74454', '74749', '75464', '71839', '72629', '67553', '74455', '72935', '72396', '72633', '68646', '73685', '76510', '72118'}]
filtre texte ok ✅ 
construction de la requete --> ✅   entreprise
filtre titre ok ✅ 
documents chercher ---> ✅  
	 {'74450': 9, '74167': 9, '72933': 9, '73875': 8, '72629': 8, '74748': 7, '73879': 7, '7